# Archived coursework source
Recovered from the author's Google Drive on 2026-10-27.
This historical code is preserved for provenance, not the supported entry point.
Outputs, images, and Colab metadata were removed. The EMNIST export/download cells were omitted.
For EMNIST, the original loop used the official test split for model selection and did not correct image orientation.
Use ../Handwritten_Text_Recognition.ipynb or the htr package for the revised workflow.


# Распознавание рукописных цифр (MNIST)


- обучает модель на наборе MNIST;
- принимает **локальный файл** или **ссылку на картинку**;
- подготавливает изображение под формат MNIST;
- распознаёт цифру и показывает вероятности.

In [ ]:

!pip install tensorflow pillow numpy

In [ ]:
import io
import os
import urllib.request

import numpy as np
from PIL import Image, ImageChops, ImageOps
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
MODEL_PATH = 'mnist_cnn.keras'
IMG_SIZE = 28

In [ ]:
def build_model():
    model = keras.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, 3, activation='relu'),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dropout(0.25),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.25),
        layers.Dense(10, activation='softmax'),
    ])
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

In [ ]:
def train_and_save(model_path=MODEL_PATH, epochs=3):
    print('[INFO] Загружаю MNIST...')
    (x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

    x_train = x_train.astype('float32') / 255.0
    x_test = x_test.astype('float32') / 255.0
    x_train = np.expand_dims(x_train, -1)
    x_test = np.expand_dims(x_test, -1)

    model = build_model()
    print('[INFO] Обучаю модель...')
    model.fit(
        x_train,
        y_train,
        validation_split=0.1,
        epochs=epochs,
        batch_size=128,
        verbose=2,
    )

    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    print(f'[INFO] Точность на тесте: {acc:.4f}')
    print(f'[INFO] Ошибка на тесте: {loss:.4f}')

    model.save(model_path)
    print(f'[INFO] Модель сохранена в: {model_path}')

In [ ]:
def load_image_from_source(source):
    if source.startswith('http://') or source.startswith('https://'):
        with urllib.request.urlopen(source) as response:
            data = response.read()
        img = Image.open(io.BytesIO(data))
    else:
        img = Image.open(source)
    return img

In [ ]:
def trim_to_content(img):
    bg = Image.new(img.mode, img.size, 0)
    diff = ImageChops.difference(img, bg)
    bbox = diff.getbbox()
    if bbox is None:
        return img
    return img.crop(bbox)

In [ ]:
def preprocess_image(source):
    img = load_image_from_source(source).convert('L')

    # Делаем изображение похожим на MNIST:
    # светлая цифра на тёмном фоне.
    arr = np.array(img)
    if arr.mean() > 127:
        img = ImageOps.invert(img)

    # Упрощаем фон/цифру.
    img = img.point(lambda p: 255 if p > 100 else 0)

    # Обрезаем пустые поля.
    img = trim_to_content(img)

    # Вписываем цифру примерно в 20x20 и центрируем на 28x28.
    img.thumbnail((20, 20))
    canvas = Image.new('L', (IMG_SIZE, IMG_SIZE), 0)
    x = (IMG_SIZE - img.width) // 2
    y = (IMG_SIZE - img.height) // 2
    canvas.paste(img, (x, y))

    x_arr = np.array(canvas).astype('float32') / 255.0
    x_arr = np.expand_dims(x_arr, axis=(0, -1))
    return x_arr, canvas

In [ ]:
def predict_digit(source, model_path=MODEL_PATH, show_images=True):
    if not os.path.exists(model_path):
        print('[INFO] Готовой модели нет. Сначала обучаю новую модель...')
        train_and_save(model_path=model_path)

    model = keras.models.load_model(model_path)
    original = load_image_from_source(source)
    x_arr, processed = preprocess_image(source)

    prediction = model.predict(x_arr, verbose=0)[0]
    digit = int(np.argmax(prediction))
    confidence = float(np.max(prediction))

    print(f'Распознанная цифра: {digit}')
    print(f'Уверенность модели: {confidence:.2%}')
    print('Вероятности по классам:')
    for i, p in enumerate(prediction):
        print(f'  {i}: {p:.2%}')

    if show_images:
        plt.figure(figsize=(4, 4))
        plt.imshow(original, cmap='gray')
        plt.title('Исходное изображение')
        plt.axis('off')
        plt.show()

        plt.figure(figsize=(4, 4))
        plt.imshow(processed, cmap='gray')
        plt.title('Подготовленное изображение 28x28')
        plt.axis('off')
        plt.show()

    return digit, confidence, prediction, processed

## 1) Обучение модели



In [ ]:
train_and_save(epochs=8)

## 2) Распознавание по локальному файлу




In [ ]:
# Пример: загрузи свой файл и укажи путь к нему
source = '/content/Снимок экрана 2026-03-25 в 20.24.10.png'
predict_digit(source)

## 3) Распознавание по ссылке



In [ ]:
# Пример URL
# source = 'https://example.com/digit.png'
# predict_digit(source)

## 4) Загрузка файла прямо в Colab через окно выбора




In [ ]:
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]
print('Загружен файл:', filename)
predict_digit(filename)